In [1]:
import os
BASE_DIR = "\\notebooks\\experiments\\params_prediction"
if os.getcwd().endswith(BASE_DIR):
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

In [2]:
from src.models.model_handler import save_model, load_model, get_class_def_from_notebook
from src.models.testing_utils import evaluate_continuous, predict_continuous
from src.data.DatasetGenerator import DatasetGenerator
from src.data.preprocessing import min_max_normalize
from src.models.training_utils import train_model
from src.utils.helpers import load_param

from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import torch

In [3]:
# Aconseguim el dataset
num_particules = 10000
dg = DatasetGenerator(num_particules)
data = dg.generate_ContinuousTimeRandomWalk(random_params=True, num_passos=200)

In [4]:
# Aconseguim un dataframe amb les coordenades i la alpha
df = data[["coords", "attributes"]].copy()
df["alpha"] = df["attributes"].apply(lambda x: x["alpha"])
df = df[["coords", "alpha"]]

In [5]:
# Tractem les coordenades per aconseguir X_tensor
coords_list = [np.stack([row[0], row[1]], axis=1) for row in df['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1) # Per aconseguir la forma (n_samples, 2, seq_length)

In [6]:
# Tractem les coordenades per aconseguir y_tensor
max_alpha, min_alpha = df["alpha"].max(), df["alpha"].min()
alpha_series = df["alpha"].apply(lambda x: min_max_normalize(x, min_alpha, max_alpha))
labels = alpha_series.values  # Suponiendo que 'alpha' contiene los valores reales
y_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

In [7]:
# Aconseguir X_train, X_val, X_test, y_train, y_val i y_test
n, train_end, val_end = len(df), int(0.7 * len(df)), int(0.85 * len(df))
X_train, X_val, X_test = X_tensor[:train_end], X_tensor[train_end:val_end], X_tensor[val_end:]
y_train, y_val, y_test = y_tensor[:train_end], y_tensor[train_end:val_end], y_tensor[val_end:]

In [8]:
# Muntem els Data_loaders
learning_batch_size = 32
try_batch_size = 1

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=learning_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=try_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=try_batch_size, shuffle=True)

In [9]:
class ContinuousTimeRandomWalkAlphaRegressor(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        # Bloque 1
        self.first_conv = nn.Conv1d(input_channels, 32, 3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, 3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, 3, padding=1)
        
        # Bloque 2
        self.conv_32_64 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, 3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, 3, padding=1)
        
        # Bloque 3
        self.conv_64_128 = nn.Conv1d(64, 128, 3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, 3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, 3, padding=1)
        
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool1d(4)  # Corregido a 1D
        self.fc1 = nn.Linear(128*4, 256)         # Ajustada la entrada
        self.fc2 = nn.Linear(256, 1)
        
        # Opcional: Añadir BatchNorm
        self.bn32 = nn.BatchNorm1d(32)
        self.bn64 = nn.BatchNorm1d(64)
        self.bn128 = nn.BatchNorm1d(128)

        self.dropout_conv = nn.Dropout(p=0.1)   # Després de blocs convolucionals
        self.dropout_fc = nn.Dropout(p=0.5)     # Abans de la capa de sortida

    def forward(self, x):
        # Bloque 1
        a_resnet = F.relu(self.first_conv(x))
        a = F.relu(self.conv1_32(a_resnet))
        a = F.relu(self.conv2_32(a))
        a = a + a_resnet
        a = self.dropout_conv(a)  # Dropout després del bloc residual
        a = self.max_pool(a)

        # Bloque 2
        b_resnet = F.relu(self.conv_32_64(a))
        b = F.relu(self.conv1_64(b_resnet))
        b = F.relu(self.conv2_64(b))
        b = b + b_resnet
        b = self.dropout_conv(b)
        b = self.max_pool(b)

        # Bloque 3
        c_resnet = F.relu(self.conv_64_128(b))
        c = F.relu(self.conv1_128(c_resnet))
        c = F.relu(self.conv2_128(c))
        c = c + c_resnet
        c = self.dropout_conv(c)
        c = self.avg_pool(c)

        # Clasificación
        c = c.view(c.size(0), -1)
        c = F.relu(self.fc1(c))
        c = self.dropout_fc(c)  # Dropout abans de la capa de sortida
        c = self.fc2(c)
        c = torch.tanh(c)
        out = (c + 1) / 2
        
        return out

input_channels = 2

# Inicialitzem el model d'aquesta manera per poder després guardar facilment els paràmetres
model_params = {
    "input_channels": input_channels
}

model = ContinuousTimeRandomWalkAlphaRegressor(**model_params)

In [10]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3),
    mode='min',
    factor=0.1,
    patience=2,
    # verbose=True <--- Deprecated
)

best_model = train_model(
    model=model,
    criterion=nn.MSELoss(),
    optimizer=scheduler.optimizer,
    num_epochs=15,
    train_loader=train_loader,
    val_loader=val_loader,
    scheduler=scheduler,
    print_epoch=True,
    patience=5
)

Epoch 1/15
Train MSE Loss: 0.0871 | Val MSE Loss: 0.0829
Epoch 2/15
Train MSE Loss: 0.0828 | Val MSE Loss: 0.0827
Epoch 3/15
Train MSE Loss: 0.0826 | Val MSE Loss: 0.0826
Epoch 4/15
Train MSE Loss: 0.0823 | Val MSE Loss: 0.0828
Epoch 5/15
Train MSE Loss: 0.0822 | Val MSE Loss: 0.0828
Epoch 6/15
Train MSE Loss: 0.0821 | Val MSE Loss: 0.0828
Epoch 7/15
Train MSE Loss: 0.0819 | Val MSE Loss: 0.0828
Epoch 8/15
Train MSE Loss: 0.0817 | Val MSE Loss: 0.0828
Early stopping!


In [11]:
evaluate_continuous(best_model, X_test, y_test)

Test MSE: 0.0827


0.08274244517087936

In [12]:
predict_continuous(best_model, X_test, y_test, num_samples=10)

Predicciones:
Ejemplo 1: Predicho = 0.49900561571121216, Real = [0.10554325]
Ejemplo 2: Predicho = 0.45432382822036743, Real = [0.06158066]
Ejemplo 3: Predicho = 0.47570884227752686, Real = [0.37722105]
Ejemplo 4: Predicho = 0.4880419075489044, Real = [0.4586813]
Ejemplo 5: Predicho = 0.4885452389717102, Real = [0.49858534]
Ejemplo 6: Predicho = 0.4777332544326782, Real = [0.08192521]
Ejemplo 7: Predicho = 0.47682127356529236, Real = [0.89745086]
Ejemplo 8: Predicho = 0.4964700937271118, Real = [0.51119274]
Ejemplo 9: Predicho = 0.5131231546401978, Real = [0.06834113]
Ejemplo 10: Predicho = 0.535377025604248, Real = [0.84847146]


In [13]:
SAVE_PATH = "../../../models/trained_models/"

In [14]:
save_model(model_instance=best_model,
           model_source=get_class_def_from_notebook(best_model.__class__.__name__),
           attributes=model_params,
           input_channels=input_channels,
           path=load_param("path", "model>trained")
           )

In [15]:
prova_carrega_model=load_model(
    load_param("path", "model>trained") +
    best_model.__class__.__name__ +
    ".pth"
)

evaluate_continuous(prova_carrega_model, X_test, y_test)
predict_continuous(prova_carrega_model, X_test, y_test, num_samples=10)

Test MSE: 0.0827
Predicciones:
Ejemplo 1: Predicho = 0.49900561571121216, Real = [0.10554325]
Ejemplo 2: Predicho = 0.45432382822036743, Real = [0.06158066]
Ejemplo 3: Predicho = 0.47570884227752686, Real = [0.37722105]
Ejemplo 4: Predicho = 0.4880419075489044, Real = [0.4586813]
Ejemplo 5: Predicho = 0.4885452389717102, Real = [0.49858534]
Ejemplo 6: Predicho = 0.4777332544326782, Real = [0.08192521]
Ejemplo 7: Predicho = 0.47682127356529236, Real = [0.89745086]
Ejemplo 8: Predicho = 0.4964700937271118, Real = [0.51119274]
Ejemplo 9: Predicho = 0.5131231546401978, Real = [0.06834113]
Ejemplo 10: Predicho = 0.535377025604248, Real = [0.84847146]
